In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import LanceDB
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

/var/folders/74/28yb9spn53l1lxl55mx5ltr00000gn/T/ipykernel_36691/2956512285.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:

loader = PyPDFLoader("../Data/intro.pdf")
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)
chunks = text_splitter.split_documents(pages)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = LanceDB.from_documents(chunks, embeddings)

/Users/sharmistaroy/Desktop/Small RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 13:02:29.834000 36691 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5844.97it/s]


In [3]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
for d in retriever.invoke("What is this document about?"):
    print("-", d.page_content[:200].replace("\n", " "))

- Deep Convolutional Neural Networks,” in Advances in Neural Information Processing Systems, vol. 25, ed. F. Pereira, C.J. Burges, L. Bottou, and K.Q. Weinberger (Curran Associates, 2012), https://oreil
- throughout the book. Building an ML system isn’t a one-off task but an iterative process. In this chapter, we discussed the iterative process to develop an ML system that met those preceding requireme
- Complex ML systems are made up of simpler building blocks. Now that we’ve covered the high-level overview of an ML system in production, we’ll zoom in to its building blocks in the following chapters,


In [6]:
from langchain_ollama import ChatOllama

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

prompt = ChatPromptTemplate.from_template(
    """Answer the question using the context below.
If the answer isn't in the context, find out from other sources and answer.

Context:
{context}

Question: {question}"""
)

llm = ChatOllama(model="llama3.2", temperature=0)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("who is the author of this document?"))

The author of this document is not explicitly mentioned in the provided context. However, based on the references and citations provided, it appears that the document is a collection of articles and papers from various sources, including academic journals, blogs, and industry publications.

The references suggest that the document is a compilation of knowledge from experts in the field of artificial intelligence, machine learning, and data science. Some of the authors mentioned in the references include:

* Jacob Devlin
* Kenton Lee
* Kristina Toutanova
* Ashok Chandrashekar
* Fernando Amat
* Justin Basilico
* Tony Jebara
* Carlos A. Gomez-Uribe
* Neil Hunt
* Parmy Olson

However, without further information, it is difficult to determine who the primary author of the document is.
